# SocialChoice 02 - Choix Social Formel en Lean 4

**Navigation** : [<< 01-Arrow-Impossibility-Theorem.ipynb](01-Arrow-Impossibility-Theorem.ipynb) | [03-Voting-Methods.ipynb >>](03-Voting-Methods.ipynb) | [Index](../README.md)

**Autres notebooks** : [04-Computational-Aggregation-SAT-Z3.ipynb](04-Computational-Aggregation-SAT-Z3.ipynb)

**Kernel** : Python 3

---

## Introduction

Ce notebook explore la **theorie du choix social**, et plus particulierement les celebres theoremes d'impossibilite d'**Arrow** (1951), de **Sen** (1970), et le **theoreme de l'electeur median** (Black 1948, Downs 1957).

Ces theoremes demontrent des limitations fondamentales des systemes de vote et de decision collective. Leur formalisation en Lean permet de verifier rigoureusement les preuves et d'explorer les hypotheses.

### Contexte historique

| Annee | Resultat | Auteur | Impact |
|-------|----------|--------|--------|
| 1785 | Paradoxe de Condorcet | Condorcet | Cycles dans les preferences collectives |
| 1948 | Theoreme de l'electeur median | Black | Convergence vers le centre |
| 1951 | Theoreme d'impossibilite | Arrow | Prix Nobel 1972 |
| 1970 | Paradoxe liberal | Sen | Conflit liberte/efficacite |

### Objectifs d'apprentissage

1. Formaliser les preferences et ordres sociaux
2. Definir et prouver le theoreme d'Arrow
3. Explorer le theoreme de Sen
4. Comprendre le theoreme de l'electeur median

### Duree estimee : 80 minutes

---

## 1. Definitions de Base

### 1.1 Preferences

Une **preference** est une relation binaire sur un ensemble d'alternatives qui est :
- **Complete** : pour tout $x, y$, soit $x \succeq y$ soit $y \succeq x$
- **Transitive** : si $x \succeq y$ et $y \succeq z$, alors $x \succeq z$

In [1]:
-- Definitions de base pour la theorie du choix social

-- Relation de preference faible : x R y signifie "x est au moins aussi bon que y"
structure Preference (A : Type) where
  R : A → A → Prop
  complete : ∀ x y, R x y ∨ R y x
  trans : ∀ x y z, R x y → R y z → R x z

-- Preference stricte derivee : x P y ssi x R y et non(y R x)
def Preference.strict {A : Type} (p : Preference A) (x y : A) : Prop :=
  p.R x y ∧ ¬ p.R y x

-- Indifference : x I y ssi x R y et y R x
def Preference.indiff {A : Type} (p : Preference A) (x y : A) : Prop :=
  p.R x y ∧ p.R y x

#check Preference
#check @Preference.strict
#check @Preference.indiff

-- Definitions de base pour la theorie du choix social

-- Relation de preference faible : x R y signifie "x est au moins aussi bon que y"
structure Preference (A : Type) where
  R : A → A → Prop
  complete : ∀ x y, R x y ∨ R y x
  trans : ∀ x y z, R x y → R y z → R x z

-- Preference stricte derivee : x P y ssi x R y et non(y R x)
def Preference.strict {A : Type} (p : Preference A) (x y : A) : Prop :=
  p.R x y ∧ ¬ p.R y x

-- Indifference : x I y ssi x R y et y R x
def Preference.indiff {A : Type} (p : Preference A) (x y : A) : Prop :=
  p.R x y ∧ p.R y x

#check Preference
──────▶  Preference (A : Type) : Type
#check @Preference.strict
──────▶  @Preference.strict : {A : Type} → Preference A → A → A → Prop
#check @Preference.indiff
──────▶  @Preference.indiff : {A : Type} → Preference A → A → A → Prop
--% env 0

Raw input:
{"cmd": "-- Definitions de base pour la theorie du choix social\n\n-- Relation de preference faible : x R y signifie \"x est au moins aussi bon que y\"\nstructure Preference (A : Type) where\n  R : A \u2192 A \u2192 Prop\n  complete : \u2200 x y, R x y \u2228 R y x\n  trans : \u2200 x y z, R x y \u2192 R y z \u2192 R x z\n\n-- Preference stricte derivee : x P y ssi x R y et non(y R x)\ndef Preference.strict {A : Type} (p : Preference A) (x y : A) : Prop :=\n  p.R x y \u2227 \u00ac p.R y x\n\n-- Indifference : x I y ssi x R y et y R x\ndef Preference.indiff {A : Type} (p : Preference A) (x y : A) : Prop :=\n  p.R x y \u2227 p.R y x\n\n#check Preference\n#check @Preference.strict\n#check @Preference.indiff"}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 17, "column": 0},
   "endPos": {"line": 17, "column": 6},
   "data": "Preference (A : Type) : Type"},
  {"severity": "info",
   "pos": {"line": 18, "column": 0},
   "endPos": {"line": 18, "column": 6},
   "data": "@Preference.strict : {A : Type} → Preference A → A → A → Prop"},
  {"severity": "info",
   "pos": {"line": 19, "column": 0},
   "endPos": {"line": 19, "column": 6},
   "data": "@Preference.indiff : {A : Type} → Preference A → A → A → Prop"}],
 "env": 0}

### 1.2 Profil de preferences

Un **profil** est une fonction qui associe a chaque individu sa preference.

In [2]:
-- Profil de preferences : chaque individu a une preference sur les alternatives
def Profile (I A : Type) := I → Preference A

-- Fonction de bien-etre social (SWF)
-- Agregue les preferences individuelles en une preference sociale
structure SocialWelfareFunction (I A : Type) where
  f : Profile I A → Preference A

#check @Profile
#check @SocialWelfareFunction

-- Profil de preferences : chaque individu a une preference sur les alternatives
def Profile (I A : Type) := I → Preference A

-- Fonction de bien-etre social (SWF)
-- Agregue les preferences individuelles en une preference sociale
structure SocialWelfareFunction (I A : Type) where
  f : Profile I A → Preference A

#check @Profile
──────▶  Profile : Type → Type → Type
#check @SocialWelfareFunction
──────▶  SocialWelfareFunction : Type → Type → Type
--% env 1

Raw input:
{"cmd": "-- Profil de preferences : chaque individu a une preference sur les alternatives\ndef Profile (I A : Type) := I \u2192 Preference A\n\n-- Fonction de bien-etre social (SWF)\n-- Agregue les preferences individuelles en une preference sociale\nstructure SocialWelfareFunction (I A : Type) where\n  f : Profile I A \u2192 Preference A\n\n#check @Profile\n#check @SocialWelfareFunction", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 9, "column": 0},
   "endPos": {"line": 9, "column": 6},
   "data": "Profile : Type → Type → Type"},
  {"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 6},
   "data": "SocialWelfareFunction : Type → Type → Type"}],
 "env": 1}

### 1.3 Helpers pour manipuler les preferences

Ces fonctions permettent de construire des profils specifiques pour les preuves.

In [3]:
-- Mettre une alternative en tete (preferee a toutes les autres)
def makeTop {A : Type} [DecidableEq A] (p : Preference A) (a : A) : Preference A := {
  R := fun x y => if x = a then True else if y = a then False else p.R x y
  complete := by
    intro x y
    by_cases hx : x = a
    · simp [hx]
    · by_cases hy : y = a
      · simp [hx, hy]
      · simp [hx, hy]; exact p.complete x y
  trans := by
    intro x y z hxy hyz
    by_cases hx : x = a
    · simp [hx]
    · by_cases hy : y = a
      · simp [hx, hy] at hxy
      · by_cases hz : z = a
        · simp [hx, hy, hz] at hyz
        · simp [hx, hy, hz] at *
          exact p.trans x y z hxy hyz
}

-- Mettre une alternative en bas (moins preferee que toutes les autres)
def makeBot {A : Type} [DecidableEq A] (p : Preference A) (a : A) : Preference A := {
  R := fun x y => if y = a then True else if x = a then False else p.R x y
  complete := by
    intro x y
    by_cases hy : y = a
    · simp [hy]
    · by_cases hx : x = a
      · simp [hx, hy]
      · simp [hx, hy]; exact p.complete x y
  trans := by
    intro x y z hxy hyz
    by_cases hz : z = a
    · simp [hz]
    · by_cases hy : y = a
      · simp [hy, hz] at hyz
      · by_cases hx : x = a
        · simp [hx, hy, hz] at hxy
        · simp [hx, hy, hz] at *
          exact p.trans x y z hxy hyz
}

#check @makeTop
#check @makeBot

-- Mettre une alternative en tete (preferee a toutes les autres)
def makeTop {A : Type} [DecidableEq A] (p : Preference A) (a : A) : Preference A := {
  R := fun x y => if x = a then True else if y = a then False else p.R x y
  complete := by
    intro x y
    by_cases hx : x = a
    · simp [hx]
    · by_cases hy : y = a
      · simp [hx, hy]
      · simp [hx, hy]; exact p.complete x y
  trans := by
    intro x y z hxy hyz
    by_cases hx : x = a
    · simp [hx]
    · by_cases hy : y = a
      · simp [hx, hy] at hxy
      · by_cases hz : z = a
        · simp [hx, hy, hz] at hyz
                ──▶ 🟨 This simp argument is unused:
  hx

Hint: Omit it from the simp argument list.
  simp [hx̵,̵ ̵h̵y, hz] at hyz

Note: This linter can be disabled with `set_option linter.unusedSimpArgs false`
        · simp [hx, hy, hz] at *
          exact p.trans x y z hxy hyz
}

-- Mettre une alternative en bas (moins preferee que toutes les autres)
def makeBot {A : Type} [DecidableEq A] (p : Preference A) (a : A) : Preference A := {
  R := fun x y => if y = a then True else if x = a then False else p.R x y
  complete := by
    intro x y
    by_cases hy : y = a
    · simp [hy]
    · by_cases hx : x = a
      · simp [hx, hy]
      · simp [hx, hy]; exact p.complete x y
  trans := by
    intro x y z hxy hyz
    by_cases hz : z = a
    · simp [hz]
    · by_cases hy : y = a
      · simp [hy, hz] at hyz
      · by_cases hx : x = a
        · simp [hx, hy, hz] at hxy
                        ──▶ 🟨 This simp argument is unused:
  hz

Hint: Omit it from the simp argument list.
  simp [hx, hy,̵ ̵h̵z̵] at hxy

Note: This linter can be disabled with `set_option linter.unusedSimpArgs false`
        · simp [hx, hy, hz] at *
          exact p.trans x y z hxy hyz
}

#check @makeTop
──────▶  @makeTop : {A : Type} → [DecidableEq A] → Preference A → A → Preference A
#check @makeBot
──────▶  @makeBot : {A : Type} → [DecidableEq A] → Preference A → A → Preference A
--% env 2

Raw input:
{"cmd": "-- Mettre une alternative en tete (preferee a toutes les autres)\ndef makeTop {A : Type} [DecidableEq A] (p : Preference A) (a : A) : Preference A := {\n  R := fun x y => if x = a then True else if y = a then False else p.R x y\n  complete := by\n    intro x y\n    by_cases hx : x = a\n    \u00b7 simp [hx]\n    \u00b7 by_cases hy : y = a\n      \u00b7 simp [hx, hy]\n      \u00b7 simp [hx, hy]; exact p.complete x y\n  trans := by\n    intro x y z hxy hyz\n    by_cases hx : x = a\n    \u00b7 simp [hx]\n    \u00b7 by_cases hy : y = a\n      \u00b7 simp [hx, hy] at hxy\n      \u00b7 by_cases hz : z = a\n        \u00b7 simp [hx, hy, hz] at hyz\n        \u00b7 simp [hx, hy, hz] at *\n          exact p.trans x y z hxy hyz\n}\n\n-- Mettre une alternative en bas (moins preferee que toutes les autres)\ndef makeBot {A : Type} [DecidableEq A] (p : Preference A) (a : A) : Preference A := {\n  R := fun x y => if y = a then True else if x = a then False else p.R x y\n  complete := by\n    intro x y\n    by_cases hy : y = a\n    \u00b7 simp [hy]\n    \u00b7 by_cases hx : x = a\n      \u00b7 simp [hx, hy]\n      \u00b7 simp [hx, hy]; exact p.complete x y\n  trans := by\n    intro x y z hxy hyz\n    by_cases hz : z = a\n    \u00b7 simp [hz]\n    \u00b7 by_cases hy : y = a\n      \u00b7 simp [hy, hz] at hyz\n      \u00b7 by_cases hx : x = a\n        \u00b7 simp [hx, hy, hz] at hxy\n        \u00b7 simp [hx, hy, hz] at *\n          exact p.trans x y z hxy hyz\n}\n\n#check @makeTop\n#check @makeBot", "env": 1}
Raw output:
{"messages":
 [{"severity": "warning",
   "pos": {"line": 18, "column": 16},
   "endPos": {"line": 18, "column": 18},
   "data":
   "This simp argument is unused:\n  hx\n\nHint: Omit it from the simp argument list.\n  simp [hx̵,̵ ̵h̵y, hz] at hyz\n\nNote: This linter can be disabled with `set_option linter.unusedSimpArgs false`"},
  {"severity": "warning",
   "pos": {"line": 40, "column": 24},
   "endPos": {"line": 40, "column": 26},
   "data":
   "This simp argument is u

---

## 2. Axiomes d'Arrow

Le theoreme d'Arrow repose sur trois axiomes qu'on pourrait considerer comme "raisonnables" pour un systeme de vote.

### 2.1 Pareto faible (P)

Si TOUS les individus preferent strictement x a y, alors la societe prefere strictement x a y.

In [4]:
-- Axiome de Pareto faible
def WeakPareto {I A : Type} (swf : SocialWelfareFunction I A) : Prop :=
  ∀ (prefs : Profile I A) (x y : A),
    (∀ i : I, (prefs i).strict x y) →
    (swf.f prefs).strict x y

#check @WeakPareto

-- Axiome de Pareto faible
def WeakPareto {I A : Type} (swf : SocialWelfareFunction I A) : Prop :=
  ∀ (prefs : Profile I A) (x y : A),
    (∀ i : I, (prefs i).strict x y) →
    (swf.f prefs).strict x y

#check @WeakPareto
──────▶  @WeakPareto : {I A : Type} → SocialWelfareFunction I A → Prop
--% env 3

Raw input:
{"cmd": "-- Axiome de Pareto faible\ndef WeakPareto {I A : Type} (swf : SocialWelfareFunction I A) : Prop :=\n  \u2200 (prefs : Profile I A) (x y : A),\n    (\u2200 i : I, (prefs i).strict x y) \u2192\n    (swf.f prefs).strict x y\n\n#check @WeakPareto", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 6},
   "data": "@WeakPareto : {I A : Type} → SocialWelfareFunction I A → Prop"}],
 "env": 3}

### 2.2 Independance des alternatives non pertinentes (IIA)

La preference sociale entre x et y ne depend QUE des preferences individuelles entre x et y.

In [5]:
-- Independance des Alternatives Irrelevantes (IIA)
def IIA {I A : Type} (swf : SocialWelfareFunction I A) : Prop :=
  ∀ (prefs prefs2 : Profile I A) (x y : A),
    (∀ i : I, (prefs i).R x y ↔ (prefs2 i).R x y) →
    (∀ i : I, (prefs i).R y x ↔ (prefs2 i).R y x) →
    ((swf.f prefs).R x y ↔ (swf.f prefs2).R x y)

#check @IIA

-- Independance des Alternatives Irrelevantes (IIA)
def IIA {I A : Type} (swf : SocialWelfareFunction I A) : Prop :=
  ∀ (prefs prefs2 : Profile I A) (x y : A),
    (∀ i : I, (prefs i).R x y ↔ (prefs2 i).R x y) →
    (∀ i : I, (prefs i).R y x ↔ (prefs2 i).R y x) →
    ((swf.f prefs).R x y ↔ (swf.f prefs2).R x y)

#check @IIA
──────▶  @IIA : {I A : Type} → SocialWelfareFunction I A → Prop
--% env 4

Raw input:
{"cmd": "-- Independance des Alternatives Irrelevantes (IIA)\ndef IIA {I A : Type} (swf : SocialWelfareFunction I A) : Prop :=\n  \u2200 (prefs prefs2 : Profile I A) (x y : A),\n    (\u2200 i : I, (prefs i).R x y \u2194 (prefs2 i).R x y) \u2192\n    (\u2200 i : I, (prefs i).R y x \u2194 (prefs2 i).R y x) \u2192\n    ((swf.f prefs).R x y \u2194 (swf.f prefs2).R x y)\n\n#check @IIA", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 6},
   "data": "@IIA : {I A : Type} → SocialWelfareFunction I A → Prop"}],
 "env": 4}

### 2.3 Non-dictature

Il n'existe PAS d'individu d tel que pour TOUT profil et TOUTE paire, si d prefere strictement x a y, alors la societe aussi.

In [6]:
-- Dictateur : un individu dont la preference stricte est toujours suivie
def IsDictator {I A : Type} (swf : SocialWelfareFunction I A) (d : I) : Prop :=
  ∀ (prefs : Profile I A) (x y : A),
    (prefs d).strict x y → (swf.f prefs).strict x y

-- Non-dictature : il n'existe pas de dictateur
def NonDictatorial {I A : Type} (swf : SocialWelfareFunction I A) : Prop :=
  ¬ ∃ d : I, IsDictator swf d

#check @IsDictator
#check @NonDictatorial

-- Dictateur : un individu dont la preference stricte est toujours suivie
def IsDictator {I A : Type} (swf : SocialWelfareFunction I A) (d : I) : Prop :=
  ∀ (prefs : Profile I A) (x y : A),
    (prefs d).strict x y → (swf.f prefs).strict x y

-- Non-dictature : il n'existe pas de dictateur
def NonDictatorial {I A : Type} (swf : SocialWelfareFunction I A) : Prop :=
  ¬ ∃ d : I, IsDictator swf d

#check @IsDictator
──────▶  @IsDictator : {I A : Type} → SocialWelfareFunction I A → I → Prop
#check @NonDictatorial
──────▶  @NonDictatorial : {I A : Type} → SocialWelfareFunction I A → Prop
--% env 5

Raw input:
{"cmd": "-- Dictateur : un individu dont la preference stricte est toujours suivie\ndef IsDictator {I A : Type} (swf : SocialWelfareFunction I A) (d : I) : Prop :=\n  \u2200 (prefs : Profile I A) (x y : A),\n    (prefs d).strict x y \u2192 (swf.f prefs).strict x y\n\n-- Non-dictature : il n'existe pas de dictateur\ndef NonDictatorial {I A : Type} (swf : SocialWelfareFunction I A) : Prop :=\n  \u00ac \u2203 d : I, IsDictator swf d\n\n#check @IsDictator\n#check @NonDictatorial", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 6},
   "data": "@IsDictator : {I A : Type} → SocialWelfareFunction I A → I → Prop"},
  {"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 6},
   "data":
   "@NonDictatorial : {I A : Type} → SocialWelfareFunction I A → Prop"}],
 "env": 5}

---

## 3. Theoreme d'Arrow

### 3.1 Enonce

**Theoreme d'Arrow (1951)** : S'il y a au moins 3 alternatives et au moins 2 individus, alors toute fonction de bien-etre social satisfaisant Pareto faible et IIA est dictatoriale.

$$\text{Pareto} \land \text{IIA} \Rightarrow \neg\text{NonDictatorial}$$

In [7]:
-- Theoreme d'Arrow : toute SWF satisfaisant Pareto et IIA est dictatoriale
-- Version sketch (preuve complete: social_choice_lean/SocialChoice/Arrow.lean, theorem arrow)
-- Preuve Geanakoplos 2005: extremal_lemma -> pivot_exists -> pivot_is_dictator_except_b
--            -> partial_dictator_is_full_dictator -> arrow (0 sorry, ~950 lignes)

theorem arrow_impossibility_sketch 
    {I A : Type} 
    [DecidableEq A]
    [Inhabited I]
    [Inhabited A]
    (swf : SocialWelfareFunction I A)
    (h_pareto : WeakPareto swf)
    (h_iia : IIA swf)
    -- Hypothese de cardinalite : au moins 3 alternatives
    (h_three_alts : ∃ a b c : A, a ≠ b ∧ b ≠ c ∧ a ≠ c) :
    -- Conclusion : il existe un dictateur
    ∃ d : I, IsDictator swf d := by
  sorry

#check @arrow_impossibility_sketch

### 3.2 Lemmes cles de la preuve

La preuve classique du theoreme d'Arrow procede en plusieurs etapes.

In [8]:
-- Lemme 1 : Extremal Lemma
-- Si tous les individus placent a en position extreme, la preference sociale aussi

def isTop {A : Type} (p : Preference A) (a : A) : Prop :=
  ∀ x : A, p.R a x

def isBot {A : Type} (p : Preference A) (a : A) : Prop :=
  ∀ x : A, p.R x a

theorem extremal_lemma 
    {I A : Type} [DecidableEq A]
    (swf : SocialWelfareFunction I A)
    (h_pareto : WeakPareto swf)
    (h_iia : IIA swf)
    (prefs : Profile I A) (a : A)
    (h_extreme : ∀ i, isTop (prefs i) a ∨ isBot (prefs i) a) :
    isTop (swf.f prefs) a ∨ isBot (swf.f prefs) a := by
  sorry

#check @extremal_lemma

-- Lemme 1 : Extremal Lemma
-- Si tous les individus placent a en position extreme, la preference sociale aussi

def isTop {A : Type} (p : Preference A) (a : A) : Prop :=
  ∀ x : A, p.R a x

def isBot {A : Type} (p : Preference A) (a : A) : Prop :=
  ∀ x : A, p.R x a

theorem extremal_lemma 
        ──────────────▶ 🟨 declaration uses 'sorry'
    {I A : Type} [DecidableEq A]
    (swf : SocialWelfareFunction I A)
    (h_pareto : WeakPareto swf)
    (h_iia : IIA swf)
    (prefs : Profile I A) (a : A)
    (h_extreme : ∀ i, isTop (prefs i) a ∨ isBot (prefs i) a) :
    isTop (swf.f prefs) a ∨ isBot (swf.f prefs) a := by
  sorry

#check @extremal_lemma
──────▶  @extremal_lemma : ∀ {I A : Type} [DecidableEq A] (swf : SocialWelfareFunction I A),
  WeakPareto swf →
    IIA swf →
      ∀ (prefs : Profile I A) (a : A),
        (∀ (i : I), isTop (prefs i) a ∨ isBot (prefs i) a) → isTop (swf.f prefs) a ∨ isBot (swf.f prefs) a
--% env 7
--% prove 0

Raw input:
{"cmd": "-- Lemme 1 : Extremal Lemma\n-- Si tous les individus placent a en position extreme, la preference sociale aussi\n\ndef isTop {A : Type} (p : Preference A) (a : A) : Prop :=\n  \u2200 x : A, p.R a x\n\ndef isBot {A : Type} (p : Preference A) (a : A) : Prop :=\n  \u2200 x : A, p.R x a\n\ntheorem extremal_lemma \n    {I A : Type} [DecidableEq A]\n    (swf : SocialWelfareFunction I A)\n    (h_pareto : WeakPareto swf)\n    (h_iia : IIA swf)\n    (prefs : Profile I A) (a : A)\n    (h_extreme : \u2200 i, isTop (prefs i) a \u2228 isBot (prefs i) a) :\n    isTop (swf.f prefs) a \u2228 isBot (swf.f prefs) a := by\n  sorry\n\n#check @extremal_lemma", "env": 6}
Raw output:
{"sorries":
 [{"proofState": 0,
   "pos": {"line": 18, "column": 2},
   "goal":
   "I A : Type\ninst✝ : DecidableEq A\nswf : SocialWelfareFunction I A\nh_pareto : WeakPareto swf\nh_iia : IIA swf\nprefs : Profile I A\na : A\nh_extreme : ∀ (i : I), isTop A (prefs i) a ∨ isBot A (prefs i) a\n⊢ isTop A (swf.f prefs) a ∨ isBot A (swf.f prefs) a",
   "endPos": {"line": 18, "column": 7}}],
 "messages":
 [{"severity": "warning",
   "pos": {"line": 10, "column": 8},
   "endPos": {"line": 10, "column": 22},
   "data": "declaration uses 'sorry'"},
  {"severity": "info",
   "pos": {"line": 20, "column": 0},
   "endPos": {"line": 20, "column": 6},
   "data":
   "@extremal_lemma : ∀ {I A : Type} [DecidableEq A] (swf : SocialWelfareFunction I A),\n  WeakPareto swf →\n    IIA swf →\n      ∀ (prefs : Profile I A) (a : A),\n        (∀ (i : I), isTop (prefs i) a ∨ isBot (prefs i) a) → isTop (swf.f prefs) a ∨ isBot (swf.f prefs) a"}],
 "env": 7}

Le **Lemme Extremal** est la première étape de la preuve : il montre que si tous les individus placent une alternative en position extrême (première ou dernière), alors la société fait de même.

Le lemme suivant introduit la notion d'**ensemble décisif** : un groupe qui peut "imposer" sa préférence sur une paire d'alternatives.

In [9]:
-- Lemme 2 : Ensemble decisif
-- Un groupe G est decisif pour (x, y) si quand tous dans G preferent x a y,
-- la societe prefere aussi x a y

def IsDecisivePred {I A : Type} (swf : SocialWelfareFunction I A) 
    (G : I → Prop) (x y : A) : Prop :=
  ∀ prefs : Profile I A,
    (∀ i : I, G i → (prefs i).strict x y) →
    (swf.f prefs).strict x y

-- Par Pareto, tous les individus ensemble sont decisifs
theorem all_decisive_pred {I A : Type} 
    (swf : SocialWelfareFunction I A)
    (h_pareto : WeakPareto swf)
    (x y : A) : 
    IsDecisivePred swf (fun _ => True) x y := by
  intro prefs h_all
  apply h_pareto
  intro i
  exact h_all i True.intro

#check @all_decisive_pred

-- Lemme 2 : Ensemble decisif
-- Un groupe G est decisif pour (x, y) si quand tous dans G preferent x a y,
-- la societe prefere aussi x a y

def IsDecisivePred {I A : Type} (swf : SocialWelfareFunction I A) 
    (G : I → Prop) (x y : A) : Prop :=
  ∀ prefs : Profile I A,
    (∀ i : I, G i → (prefs i).strict x y) →
    (swf.f prefs).strict x y

-- Par Pareto, tous les individus ensemble sont decisifs
theorem all_decisive_pred {I A : Type} 
    (swf : SocialWelfareFunction I A)
    (h_pareto : WeakPareto swf)
    (x y : A) : 
    IsDecisivePred swf (fun _ => True) x y := by
  intro prefs h_all
  apply h_pareto
  intro i
  exact h_all i True.intro

#check @all_decisive_pred
──────▶  @all_decisive_pred : ∀ {I A : Type} (swf : SocialWelfareFunction I A),
  WeakPareto swf → ∀ (x y : A), IsDecisivePred swf (fun x => True) x y
--% env 8

Raw input:
{"cmd": "-- Lemme 2 : Ensemble decisif\n-- Un groupe G est decisif pour (x, y) si quand tous dans G preferent x a y,\n-- la societe prefere aussi x a y\n\ndef IsDecisivePred {I A : Type} (swf : SocialWelfareFunction I A) \n    (G : I \u2192 Prop) (x y : A) : Prop :=\n  \u2200 prefs : Profile I A,\n    (\u2200 i : I, G i \u2192 (prefs i).strict x y) \u2192\n    (swf.f prefs).strict x y\n\n-- Par Pareto, tous les individus ensemble sont decisifs\ntheorem all_decisive_pred {I A : Type} \n    (swf : SocialWelfareFunction I A)\n    (h_pareto : WeakPareto swf)\n    (x y : A) : \n    IsDecisivePred swf (fun _ => True) x y := by\n  intro prefs h_all\n  apply h_pareto\n  intro i\n  exact h_all i True.intro\n\n#check @all_decisive_pred", "env": 7}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 22, "column": 0},
   "endPos": {"line": 22, "column": 6},
   "data":
   "@all_decisive_pred : ∀ {I A : Type} (swf : SocialWelfareFunction I A),\n  WeakPareto swf → ∀ (x y : A), IsDecisivePred swf (fun x => True) x y"}],
 "env": 8}

---

## 4. Theoreme de Sen

### 4.1 Le paradoxe liberal

Le **theoreme de Sen** (1970) montre un conflit entre **liberte individuelle** et **efficacite Pareto**.

### 4.2 Liberte minimale

In [10]:
-- Liberte minimale (Minimal Liberalism) - version bidirectionnelle (Sen 1970)
-- Chaque individu est "decisif" sur au moins une paire d'alternatives,
-- dans les DEUX directions : s'il prefere x a y, la societe aussi,
-- et s'il prefere y a x, la societe aussi.
-- Cette bidirectionnalite est essentielle pour le theoreme d'impossibilite.

def MinimalLiberalism {I A : Type} (swf : SocialWelfareFunction I A) : Prop :=
  ∀ i : I, ∃ x y : A, x ≠ y ∧
    (∀ prefs : Profile I A,
      (prefs i).strict x y → (swf.f prefs).strict x y) ∧
    (∀ prefs : Profile I A,
      (prefs i).strict y x → (swf.f prefs).strict y x)

#check @MinimalLiberalism

La **Liberte Minimale** est une condition tres faible : elle demande seulement que chaque individu ait le controle sur AU MOINS une paire d'alternatives (par exemple, ce qu'il lit dans sa sphere privee).

**Bidirectionnalite** : la definition exige que l'individu soit decisif dans les deux sens — s'il prefere x a y, la societe prefere x a y, ET s'il prefere y a x, la societe prefere y a x. C'est la formulation originale de Sen (1970), et elle est essentielle pour construire le cycle dans la preuve du theoreme d'impossibilite.

Le theoreme de Sen montre que meme cette condition minimale entre en conflit avec l'efficacite Pareto.

> **Preuve complete** : la formalisation complete du theoreme de Sen avec 0 `sorry` se trouve dans le projet Lake `social_choice_lean/` (fichiers `Sen.lean` et `Arrow.lean`).

In [11]:
-- Theoreme de Sen : impossibilite de satisfaire simultanement Pareto et Liberte
-- Pareto + Liberalisme minimal sont contradictoires
-- Preuve complete: social_choice_lean/SocialChoice/Sen.lean, theorem sen_impossibility
-- (0 sorry, ~300 lignes, construction de profil avec cycle social)

theorem sen_impossibility_sketch
    {I A : Type}
    [DecidableEq A]
    [Inhabited I] [Inhabited A]
    (swf : SocialWelfareFunction I A)
    (h_pareto : WeakPareto swf)
    (h_liberal : MinimalLiberalism swf)
    -- Hypotheses de cardinalite necessaires pour le theoreme
    (h_two_voters : ∃ i j : I, i ≠ j)
    (h_three_alts : ∃ a b c : A, a ≠ b ∧ b ≠ c ∧ a ≠ c) :
    -- Pareto et Liberalisme sont incompatibles
    False := by
  sorry

#check @sen_impossibility_sketch

L'exemple de **Lady Chatterley** est illustre dans le notebook Python compagnon (20b).

---

## 5. Theoreme de l'Electeur Median

### 5.1 Preferences unimodales (Single-peaked)

Le theoreme d'Arrow montre qu'aucune regle de vote parfaite n'existe en general. Cependant, si on **restreint le domaine** a des preferences unimodales, des resultats positifs emergent.

In [12]:
-- Preferences unimodales (single-peaked)
-- L'espace des alternatives est ordonne (ex: gauche-droite sur [0, 1])

def SinglePeaked {A : Type} (le : A → A → Prop) (p : Preference A) (peak : A) : Prop :=
  -- A gauche du pic : plus on approche du pic, mieux c'est
  (∀ x y : A, le x y → le y peak ∨ y = peak → p.R y x) ∧
  -- A droite du pic : plus on s'eloigne du pic, moins bon c'est
  (∀ x y : A, le peak x ∨ peak = x → le x y → p.R x y)

#check @SinglePeaked

-- Un profil est unimodal si tous les electeurs ont des preferences unimodales
-- Note: On utilise un ordre explicite (le) au lieu de [LinearOrder A] pour eviter Mathlib
def SinglePeakedProfile {I A : Type} (le : A → A → Prop) (prefs : Profile I A) : Prop :=
  ∀ i : I, ∃ peak : A, SinglePeaked le (prefs i) peak

#check @SinglePeakedProfile

-- Preferences unimodales (single-peaked)
-- L'espace des alternatives est ordonne (ex: gauche-droite sur [0, 1])

def SinglePeaked {A : Type} (le : A → A → Prop) (p : Preference A) (peak : A) : Prop :=
  -- A gauche du pic : plus on approche du pic, mieux c'est
  (∀ x y : A, le x y → le y peak ∨ y = peak → p.R y x) ∧
  -- A droite du pic : plus on s'eloigne du pic, moins bon c'est
  (∀ x y : A, le peak x ∨ peak = x → le x y → p.R x y)

#check @SinglePeaked
──────▶  @SinglePeaked : {A : Type} → (A → A → Prop) → Preference A → A → Prop

-- Un profil est unimodal si tous les electeurs ont des preferences unimodales
-- Note: On utilise un ordre explicite (le) au lieu de [LinearOrder A] pour eviter Mathlib
def SinglePeakedProfile {I A : Type} (le : A → A → Prop) (prefs : Profile I A) : Prop :=
  ∀ i : I, ∃ peak : A, SinglePeaked le (prefs i) peak

#check @SinglePeakedProfile
──────▶  @SinglePeakedProfile : {I A : Type} → (A → A → Prop) → Profile I A → Prop
--% env 11

Raw input:
{"cmd": "-- Preferences unimodales (single-peaked)\n-- L'espace des alternatives est ordonne (ex: gauche-droite sur [0, 1])\n\ndef SinglePeaked {A : Type} (le : A \u2192 A \u2192 Prop) (p : Preference A) (peak : A) : Prop :=\n  -- A gauche du pic : plus on approche du pic, mieux c'est\n  (\u2200 x y : A, le x y \u2192 le y peak \u2228 y = peak \u2192 p.R y x) \u2227\n  -- A droite du pic : plus on s'eloigne du pic, moins bon c'est\n  (\u2200 x y : A, le peak x \u2228 peak = x \u2192 le x y \u2192 p.R x y)\n\n#check @SinglePeaked\n\n-- Un profil est unimodal si tous les electeurs ont des preferences unimodales\n-- Note: On utilise un ordre explicite (le) au lieu de [LinearOrder A] pour eviter Mathlib\ndef SinglePeakedProfile {I A : Type} (le : A \u2192 A \u2192 Prop) (prefs : Profile I A) : Prop :=\n  \u2200 i : I, \u2203 peak : A, SinglePeaked le (prefs i) peak\n\n#check @SinglePeakedProfile", "env": 10}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 6},
   "data":
   "@SinglePeaked : {A : Type} → (A → A → Prop) → Preference A → A → Prop"},
  {"severity": "info",
   "pos": {"line": 17, "column": 0},
   "endPos": {"line": 17, "column": 6},
   "data":
   "@SinglePeakedProfile : {I A : Type} → (A → A → Prop) → Profile I A → Prop"}],
 "env": 11}

### 5.2 Theoreme de l'electeur median

**Theoreme (Black 1948)** : Si tous les electeurs ont des preferences unimodales sur un espace unidimensionnel, alors sous vote majoritaire, l'alternative preferee de l'**electeur median** est le **vainqueur de Condorcet**.

In [13]:
-- Theoreme de l'electeur median : sous preferences unimodales,
-- il existe un gagnant de Condorcet
-- Note: La preuve complete necessite Fintype + LinearOrder + cardinalite impaire.
-- Le projet social_choice_lean ne contient pas encore cette preuve (a venir).

-- Un gagnant de Condorcet : alternative que tout electeur prefere
-- aux alternatives situees du cote oppose de son pic
-- C'est la propriete structurelle qui rend le gagnant de Condorcet possible
def CondorcetWinner {I A : Type}
    (winner : A) (le : A → A → Prop) (prefs : Profile I A) : Prop :=
  ∀ i : I, ∀ peak alt : A,
    SinglePeaked le (prefs i) peak → alt ≠ winner →
      -- Si le pic est a gauche (ou egal) et alt a droite, l'electeur prefere winner
      ((le peak winner ∨ peak = winner) → le winner alt →
        (prefs i).R winner alt) ∧
      -- Si le pic est a droite (ou egal) et alt a gauche, l'electeur prefere winner
      ((le winner peak ∨ winner = peak) → le alt winner →
        (prefs i).R winner alt)

theorem median_voter_theorem_sketch
    {I A : Type} [Inhabited I]
    (le : A → A → Prop)
    (prefs : Profile I A)
    (h_single_peaked : SinglePeakedProfile le prefs) :
    -- Il existe un gagnant de Condorcet
    -- La preuve complete (construction du median) necessite Fintype
    ∃ winner : A, CondorcetWinner winner le prefs := by
  sorry

#check @median_voter_theorem_sketch

---

## 6. Exercices

### Exercice 1 : Dictature et axiomes

Montrer qu'une dictature satisfait les axiomes de Pareto faible et IIA.

### Exercice 2 : Deux alternatives

Montrer que le theoreme d'Arrow ne s'applique pas avec seulement 2 alternatives.

### Exercice 3 : Preuve de Pareto

Prouver formellement que l'ensemble de tous les individus est decisif.

---

## 7. Exercices

### Exercice 1 : Dictature et axiomes

Montrez qu'une dictature (ou un individu copie les preferences d'un electeur designe)
satisfait les axoimes de Pareto faible et IIA.
Construisez la dictature en Lean et prouvez ces deux proprietes.

**Indice** : Definissez  comme une SWF qui retourne les preferences
d'un individu fixe . Pour Pareto, utilisez le fait que si tout le monde prefere
 a , alors  aussi. Pour IIA, montrez que la restriction a une paire ne change
pas la preference du dictateur.

In [14]:
-- TODO etudiant : Definir dictatorship (SWF qui copie les preferences de d)
-- def dictatorship {I A : Type} (d : I) : SocialWelfareFunction I A := ...

-- TODO etudiant : Prouver qu'une dictature satisfait Pareto faible
-- theorem dictatorship_pareto {I A : Type} (d : I) :
--     WeakPareto (dictatorship d (I := I) (A := A)) := by ...

-- TODO etudiant : Prouver qu'une dictature satisfait IIA
-- theorem dictatorship_iia {I A : Type} (d : I) :
--     IIA (dictatorship d (I := I) (A := A)) := by ...

-- Indice: pour dictatorship, le champ f de la SWF est fun prefs => prefs d
-- Indice: pour Pareto, h_all d donne la preference du dictateur
-- Indice: pour IIA, simp only [dictatorship] puis utiliser h_xy d

pass  -- Remplacer par vos definitions et preuves

-- Solution Exercice 1 : Une dictature satisfait Pareto et IIA

-- Construire une dictature
def dictatorship {I A : Type} (d : I) : SocialWelfareFunction I A := {
  f := fun prefs => prefs d
}

-- Une dictature satisfait Pareto faible
theorem dictatorship_pareto {I A : Type} (d : I) : 
    WeakPareto (dictatorship d (I := I) (A := A)) := by
  intro prefs x y h_all
  exact h_all d

-- Une dictature satisfait IIA
theorem dictatorship_iia {I A : Type} (d : I) :
    IIA (dictatorship d (I := I) (A := A)) := by
  intro prefs prefs2 x y h_xy h_yx
  simp only [dictatorship]
  exact h_xy d

#check @dictatorship_pareto
──────▶  @dictatorship_pareto : ∀ {I A : Type} (d : I), WeakPareto (dictatorship d)
#check @dictatorship_iia
──────▶  @dictatorship_iia : ∀ {I A : Type} (d : I), IIA (dictatorship d)
--% env 13

Raw input:
{"cmd": "-- Solution Exercice 1 : Une dictature satisfait Pareto et IIA\n\n-- Construire une dictature\ndef dictatorship {I A : Type} (d : I) : SocialWelfareFunction I A := {\n  f := fun prefs => prefs d\n}\n\n-- Une dictature satisfait Pareto faible\ntheorem dictatorship_pareto {I A : Type} (d : I) : \n    WeakPareto (dictatorship d (I := I) (A := A)) := by\n  intro prefs x y h_all\n  exact h_all d\n\n-- Une dictature satisfait IIA\ntheorem dictatorship_iia {I A : Type} (d : I) :\n    IIA (dictatorship d (I := I) (A := A)) := by\n  intro prefs prefs2 x y h_xy h_yx\n  simp only [dictatorship]\n  exact h_xy d\n\n#check @dictatorship_pareto\n#check @dictatorship_iia", "env": 12}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 21, "column": 0},
   "endPos": {"line": 21, "column": 6},
   "data":
   "@dictatorship_pareto : ∀ {I A : Type} (d : I), WeakPareto (dictatorship d)"},
  {"severity": "info",
   "pos": {"line": 22, "column": 0},
   "endPos": {"line": 22, "column": 6},
   "data": "@dictatorship_iia : ∀ {I A : Type} (d : I), IIA (dictatorship d)"}],
 "env": 13}

### Exercice 2 : Deux alternatives

Expliquez pourquoi, avec exactement 2 alternatives, la regle majoritaire satisfait
Pareto, IIA et Non-dictature. Quel hypothesis du theoreme d'Arrow est relaxee ?

**Indice** : La condition |A| >= 3 est essentielle. Avec 2 alternatives,
la regle majoritaire est une SWF valide qui satisfait les trois axiomes.
Voir le notebook Python compagnon (20b) pour l'illustration.

---

# Annexe : Tour de SocialChoiceLean (Dominik Peters)

Cette section presente la librairie [SocialChoiceLean](https://github.com/dominikpeters/social-choice-lean) de Dominik Peters,
qui formalise en Lean 4 (avec Mathlib) de nombreuses regles de vote et theoremes d'impossibilite.
Elle complete les preuves du port  etendues dans ce notebook.


---

## 1. Cadre Formel

DominikPeters utilise un cadre base sur les **preferences strictes** (ordres lineaires stricts), ou chaque electeur a un classement sans ex aequo. Ce choix simplifie les definitions des marges et des axiomes par rapport aux preferences faibles (16b).

In [1]:
-- Simplified framework inspired by DominikPeters/SocialChoiceLean
-- Full formalization (with Mathlib): social_choice_lean_peters/PetersTour.lean

-- Strict preference: total strict order
-- Replaces Mathlib's LinearOrder for kernel compatibility
structure StrictPref (A : Type) where
  lt : A → A → Prop
  irrefl : ∀ x, ¬ lt x x
  trans : ∀ x y z, lt x y → lt y z → lt x z
  conn : ∀ x y, x ≠ y → lt x y ∨ lt y x

-- Voting profile: each voter has a strict preference
def VotingProfile (V A : Type) := V → StrictPref A

-- Voting rule: maps profile to a set of winners (as predicate)
def VotingRule (V A : Type) := VotingProfile V A → (A → Prop)

-- A rule is resolute if it always selects exactly one winner
def IsResolute {V A : Type} (f : VotingRule V A) : Prop :=
  ∀ P, ∃ c : A, f P c ∧ ∀ d, f P d → d = c

#check @StrictPref
#check @VotingProfile
#check @VotingRule
#check @IsResolute

-- Simplified framework inspired by DominikPeters/SocialChoiceLean
-- Full formalization (with Mathlib): social_choice_lean_peters/PetersTour.lean

-- Strict preference: total strict order
-- Replaces Mathlib's LinearOrder for kernel compatibility
structure StrictPref (A : Type) where
  lt : A → A → Prop
  irrefl : ∀ x, ¬ lt x x
────────────────▶ ❌ expected token
  trans : ∀ x y z, lt x y → lt y z → lt x z
  conn : ∀ x y, x ≠ y → lt x y ∨ lt y x

-- Voting profile: each voter has a strict preference
def VotingProfile (V A : Type) := V → StrictPref A
                                      ──────────▶ ❌ Unknown identifier `StrictPref`

-- Voting rule: maps profile to a set of winners (as predicate)
def VotingRule (V A : Type) := VotingProfile V A → (A → Prop)
                               ─────────────▶ ❌ Unknown identifier `VotingProfile`

-- A rule is resolute if it always selects exactly one winner
def IsResolute {V A : Type} (f : VotingRule V A) : Prop :=
                                 ──────────────▶ ❌ Unknown constant `CoeFun`
  ∀ P, ∃ c : A, f P c ∧ ∀ d, f P d → d = c
───────▶ ❌ expected token

#check @StrictPref
        ──────────▶ ❌ Unknown identifier `StrictPref`
#check @VotingProfile
        ─────────────▶ ❌ Unknown identifier `VotingProfile`
#check @VotingRule
        ──────────▶ ❌ Unknown identifier `VotingRule`
#check @IsResolute
        ──────────▶ ❌ Unknown identifier `IsResolute`
--% env 0

Raw input:
{"cmd": "-- Simplified framework inspired by DominikPeters/SocialChoiceLean\n-- Full formalization (with Mathlib): social_choice_lean_peters/PetersTour.lean\n\n-- Strict preference: total strict order\n-- Replaces Mathlib's LinearOrder for kernel compatibility\nstructure StrictPref (A : Type) where\n  lt : A \u2192 A \u2192 Prop\n  irrefl : \u2200 x, \u00ac lt x x\n  trans : \u2200 x y z, lt x y \u2192 lt y z \u2192 lt x z\n  conn : \u2200 x y, x \u2260 y \u2192 lt x y \u2228 lt y x\n\n-- Voting profile: each voter has a strict preference\ndef VotingProfile (V A : Type) := V \u2192 StrictPref A\n\n-- Voting rule: maps profile to a set of winners (as predicate)\ndef VotingRule (V A : Type) := VotingProfile V A \u2192 (A \u2192 Prop)\n\n-- A rule is resolute if it always selects exactly one winner\ndef IsResolute {V A : Type} (f : VotingRule V A) : Prop :=\n  \u2200 P, \u2203 c : A, f P c \u2227 \u2200 d, f P d \u2192 d = c\n\n#check @StrictPref\n#check @VotingProfile\n#check @VotingRule\n#check @IsResolute"}
Raw output:
{"messages":
 [{"severity": "error",
   "pos": {"line": 8, "column": 16},
   "endPos": null,
   "data": "expected token"},
  {"severity": "error",
   "pos": {"line": 13, "column": 38},
   "endPos": {"line": 13, "column": 48},
   "data": "Unknown identifier `StrictPref`"},
  {"severity": "error",
   "pos": {"line": 16, "column": 31},
   "endPos": {"line": 16, "column": 44},
   "data": "Unknown identifier `VotingProfile`"},
  {"severity": "error",
   "pos": {"line": 20, "column": 7},
   "endPos": null,
   "data": "expected token"},
  {"severity": "error",
   "pos": {"line": 19, "column": 33},
   "endPos": {"line": 19, "column": 47},
   "data": "Unknown constant `CoeFun`"},
  {"severity": "error",
   "pos": {"line": 22, "column": 8},
   "endPos": {"line": 22, "column": 18},
   "data": "Unknown identifier `StrictPref`"},
  {"severity": "error",
   "pos": {"line": 23, "column": 8},
   "endPos": {"line": 23, "column": 21},
   "data": "Unknown identifier `VotingProfile`"},
  {"severity": "error",
   "pos": {"line": 24, "column": 8},
   "endPos": {"line": 24, "column": 18},
   "data": "Unknown identifier `VotingRule`"},
  {"severity": "error",
   "pos": {"line": 25, "column": 8},
   "endPos": {"line": 25, "column": 18},
   "data": "Unknown identifier `IsResolute`"}],
 "env": 0}

### Interpretation

| Concept | Peters (Mathlib) | Ce notebook (simplifie) |
|---------|-------------------|-------------------------|
| Preference stricte | `LinearOrder A` (classe) | `StrictPref A` (structure) |
| Profil | `Profile V A [Fintype V]` | `VotingProfile V A = V -> StrictPref A` |
| Regle | `VotingRule` (polymorphe Fintype) | `VotingRule V A = Profile -> (A -> Prop)` |
| Resolutude | `Resolute f` | `IsResolute f` |

Dans Peters, `LinearOrder` fournit `lt` automatiquement. Ici, on le definit explicitement avec `irrefl`, `trans`, et `conn` (totalite pour les elements distincts).

---

## 2. Marges et Vainqueur de Condorcet

La **marge** d'un candidat sur un autre mesure l'ecart de votes : combien d'electeurs preferent a a b, moins combien preferent b a a. Un **vainqueur de Condorcet** bat tous les autres par marge positive.

In [2]:
-- Margin function: pairwise difference
-- In Peters: (Finset.univ.filter (fun v => Prefers P v a b)).card
--   minus the reverse. Here: abstract signature.
abbrev MarginFun (A : Type) := A → A → Int

-- Condorcet winner: beats every other by positive margin
def IsCondorcetWinner {A : Type} (m : MarginFun A) (c : A) : Prop :=
  ∀ d, d ≠ c → m c d > 0

-- Condorcet consistency: the rule selects the Condorcet winner uniquely
def CondorcetConsistent {V A : Type} (f : VotingRule V A)
    (margin : VotingProfile V A → MarginFun A) : Prop :=
  ∀ P c, IsCondorcetWinner (margin P) c →
    f P c ∧ ∀ d, d ≠ c → ¬ f P d

#check @MarginFun
#check @IsCondorcetWinner
#check @CondorcetConsistent

-- Margin function: pairwise difference
-- In Peters: (Finset.univ.filter (fun v => Prefers P v a b)).card
--   minus the reverse. Here: abstract signature.
abbrev MarginFun (A : Type) := A → A → Int
                                       ───▶ ❌ Unknown identifier `Int`

-- Condorcet winner: beats every other by positive margin
def IsCondorcetWinner {A : Type} (m : MarginFun A) (c : A) : Prop :=
                                      ───────────▶ ❌ Unknown constant `CoeFun`
  ∀ d, d ≠ c → m c d > 0
─────────▶ ❌ expected token

-- Condorcet consistency: the rule selects the Condorcet winner uniquely
def CondorcetConsistent {V A : Type} (f : VotingRule V A)
                                          ──────────────▶ ❌ Unknown constant `CoeFun`
    (margin : VotingProfile V A → MarginFun A) : Prop :=
  ∀ P c, IsCondorcetWinner (margin P) c →
    f P c ∧ ∀ d, d ≠ c → ¬ f P d
──────────▶ ❌ expected token

#check @MarginFun
        ─────────▶ ❌ Unknown identifier `MarginFun`
#check @IsCondorcetWinner
        ─────────────────▶ ❌ Unknown identifier `IsCondorcetWinner`
#check @CondorcetConsistent
        ───────────────────▶ ❌ Unknown identifier `CondorcetConsistent`
--% env 1

Raw input:
{"cmd": "-- Margin function: pairwise difference\n-- In Peters: (Finset.univ.filter (fun v => Prefers P v a b)).card\n--   minus the reverse. Here: abstract signature.\nabbrev MarginFun (A : Type) := A \u2192 A \u2192 Int\n\n-- Condorcet winner: beats every other by positive margin\ndef IsCondorcetWinner {A : Type} (m : MarginFun A) (c : A) : Prop :=\n  \u2200 d, d \u2260 c \u2192 m c d > 0\n\n-- Condorcet consistency: the rule selects the Condorcet winner uniquely\ndef CondorcetConsistent {V A : Type} (f : VotingRule V A)\n    (margin : VotingProfile V A \u2192 MarginFun A) : Prop :=\n  \u2200 P c, IsCondorcetWinner (margin P) c \u2192\n    f P c \u2227 \u2200 d, d \u2260 c \u2192 \u00ac f P d\n\n#check @MarginFun\n#check @IsCondorcetWinner\n#check @CondorcetConsistent", "env": 0}
Raw output:
{"messages":
 [{"severity": "error",
   "pos": {"line": 4, "column": 39},
   "endPos": {"line": 4, "column": 42},
   "data": "Unknown identifier `Int`"},
  {"severity": "error",
   "pos": {"line": 7, "column": 38},
   "endPos": {"line": 7, "column": 49},
   "data": "Unknown constant `CoeFun`"},
  {"severity": "error",
   "pos": {"line": 8, "column": 9},
   "endPos": null,
   "data": "expected token"},
  {"severity": "error",
   "pos": {"line": 11, "column": 42},
   "endPos": {"line": 11, "column": 56},
   "data": "Unknown constant `CoeFun`"},
  {"severity": "error",
   "pos": {"line": 14, "column": 10},
   "endPos": null,
   "data": "expected token"},
  {"severity": "error",
   "pos": {"line": 16, "column": 8},
   "endPos": {"line": 16, "column": 17},
   "data": "Unknown identifier `MarginFun`"},
  {"severity": "error",
   "pos": {"line": 17, "column": 8},
   "endPos": {"line": 17, "column": 25},
   "data": "Unknown identifier `IsCondorcetWinner`"},
  {"severity": "error",
   "pos": {"line": 18, "column": 8},
   "endPos": {"line": 18, "column": 27},
   "data": "Unknown identifier `CondorcetConsistent`"}],
 "env": 1}

### Interpretation

La **marge** est l'outil central du framework de Peters. Elle permet de definir :

- **Condorcet winner** : candidat avec marge positive sur tous les autres
- **Condorcet consistency** : la regle Selectionne toujours le Condorcet winner

Le concept de marge est intuitif : si 7 electeurs preferent A a B et 3 preferent B a A, la marge de A sur B est +4. Un Condorcet winner a une marge positive contre chaque adversaire.

> **Note technique** : dans la formalisation complete, la marge utilise `Finset.card` (nombre d'electeurs preferant a a b). Notre version simplifiee prend la marge comme fonction abstraite.

---

## 3. Axiomes de Vote

DominikPeters definit et utilise 15+ axiomes pour classifier les regles de vote. Chaque axiome exprime une propriete desirable : equite, robustesse, non-manipulabilite.

In [3]:
-- Pareto Efficiency: if all prefer c to d, d cannot win
def SatisfiesPareto {V A : Type} (f : VotingRule V A) : Prop :=
  ∀ P c d, (∀ v, (P v).lt c d) → ¬ f P d

-- Unanimity: if all rank c first, c must win
def SatisfiesUnanimity {V A : Type} (f : VotingRule V A) : Prop :=
  ∀ P c, (∀ v d, d ≠ c → (P v).lt c d) → f P c

-- Strategyproof (resolute): no voter can improve outcome by misreporting
-- The voter's true preferences are P v, but they report 'fake' instead
def IsStrategyproof {V A : Type} [DecidableEq A] (f : VotingRule V A) : Prop :=
  ∀ (P : VotingProfile V A) (v : V) (fake : StrictPref A),
    let P' := fun w : V => if w = v then fake else P w
    ¬ ∃ cr cf : A,
      f P cr ∧ f P' cf ∧ (P v).lt cf cr

#check @SatisfiesPareto
#check @SatisfiesUnanimity
#check @IsStrategyproof

-- Pareto Efficiency: if all prefer c to d, d cannot win
def SatisfiesPareto {V A : Type} (f : VotingRule V A) : Prop :=
                                      ──────────────▶ ❌ Unknown constant `CoeFun`
  ∀ P c d, (∀ v, (P v).lt c d) → ¬ f P d
─────────────────────────────────▶ ❌ expected token

-- Unanimity: if all rank c first, c must win
def SatisfiesUnanimity {V A : Type} (f : VotingRule V A) : Prop :=
                                         ──────────────▶ ❌ Unknown constant `CoeFun`
  ∀ P c, (∀ v d, d ≠ c → (P v).lt c d) → f P c
───────────────────▶ ❌ expected token

-- Strategyproof (resolute): no voter can improve outcome by misreporting
-- The voter's true preferences are P v, but they report 'fake' instead
def IsStrategyproof {V A : Type} [DecidableEq A] (f : VotingRule V A) : Prop :=
  ∀ (P : VotingProfile V A) (v : V) (fake : StrictPref A),
    let P' := fun w : V => if w = v then fake else P w
                          ───▶ ❌ unexpected token 'if'; expected term
    ¬ ∃ cr cf : A,
      f P cr ∧ f P' cf ∧ (P v).lt cf cr

#check @SatisfiesPareto
        ───────────────▶ ❌ Unknown identifier `SatisfiesPareto`
#check @SatisfiesUnanimity
        ──────────────────▶ ❌ Unknown identifier `SatisfiesUnanimity`
#check @IsStrategyproof
        ───────────────▶ ❌ Unknown identifier `IsStrategyproof`
--% env 2

Raw input:
{"cmd": "-- Pareto Efficiency: if all prefer c to d, d cannot win\ndef SatisfiesPareto {V A : Type} (f : VotingRule V A) : Prop :=\n  \u2200 P c d, (\u2200 v, (P v).lt c d) \u2192 \u00ac f P d\n\n-- Unanimity: if all rank c first, c must win\ndef SatisfiesUnanimity {V A : Type} (f : VotingRule V A) : Prop :=\n  \u2200 P c, (\u2200 v d, d \u2260 c \u2192 (P v).lt c d) \u2192 f P c\n\n-- Strategyproof (resolute): no voter can improve outcome by misreporting\n-- The voter's true preferences are P v, but they report 'fake' instead\ndef IsStrategyproof {V A : Type} [DecidableEq A] (f : VotingRule V A) : Prop :=\n  \u2200 (P : VotingProfile V A) (v : V) (fake : StrictPref A),\n    let P' := fun w : V => if w = v then fake else P w\n    \u00ac \u2203 cr cf : A,\n      f P cr \u2227 f P' cf \u2227 (P v).lt cf cr\n\n#check @SatisfiesPareto\n#check @SatisfiesUnanimity\n#check @IsStrategyproof", "env": 1}
Raw output:
{"messages":
 [{"severity": "error",
   "pos": {"line": 3, "column": 33},
   "endPos": null,
   "data": "expected token"},
  {"severity": "error",
   "pos": {"line": 2, "column": 38},
   "endPos": {"line": 2, "column": 52},
   "data": "Unknown constant `CoeFun`"},
  {"severity": "error",
   "pos": {"line": 7, "column": 19},
   "endPos": null,
   "data": "expected token"},
  {"severity": "error",
   "pos": {"line": 6, "column": 41},
   "endPos": {"line": 6, "column": 55},
   "data": "Unknown constant `CoeFun`"},
  {"severity": "error",
   "pos": {"line": 13, "column": 26},
   "endPos": {"line": 13, "column": 29},
   "data": "unexpected token 'if'; expected term"},
  {"severity": "error",
   "pos": {"line": 17, "column": 8},
   "endPos": {"line": 17, "column": 23},
   "data": "Unknown identifier `SatisfiesPareto`"},
  {"severity": "error",
   "pos": {"line": 18, "column": 8},
   "endPos": {"line": 18, "column": 26},
   "data": "Unknown identifier `SatisfiesUnanimity`"},
  {"severity": "error",
   "pos": {"line": 19, "column": 8},
   "endPos": {"line": 19, "column": 23},
   "data": "Unknown identifier `IsStrategyproof`"}],
 "env": 2}

### Interpretation des axiomes

| Axiome | Intuition | Exigence |
|--------|-----------|----------|
| **Pareto** | Unanimite : si tous preferent c a d, d est exclu | Minimal pour toute regle raisonnable |
| **Unanimite** | Si tous classent c premier, c doit gagner | Plus fort que Pareto |
| **Non-manipulabilite** | Aucun electeur ne peut ameliorer le resultat en mentant | Equilibre strategique |

Ces trois axiomes semblent naturels et faibles. Pourtant, le theoreme de Gibbard-Satterthwaite montre qu'ils sont **incompatibles** avec la non-dictature pour >= 3 candidats.

---

## 4. Theoreme de Gibbard-Satterthwaite

Le resultat central de la theorie du choix social : toute regle de vote resolutive, unanime et non-manipulable (pour >= 3 candidats) doit etre une dictature. Ce theoreme montre que **toute election est manipulable** (sauf dictature).

La preuve formelle dans Peters fait 5 fichiers Lean (BaseCase + Common + InductionStepCase1 + InductionStepCase2 + Main), avec une induction forte sur le nombre d'electeurs.

In [4]:
-- Dictatorial (for resolute rules): one voter's preference determines the winner
def IsDictatorialRule {V A : Type} (f : VotingRule V A) : Prop :=
  ∃ d : V, ∀ P c, f P c → ∀ d' : A, d' ≠ c → (P d).lt c d'

/-- **Gibbard-Satterthwaite (1973/1975)**
    Toute regle resolutive, unanime et non-manipulable (>= 3 candidats)
    est dictatoriale.

    Preuve par induction forte sur le nombre d'electeurs :
    - Base (1 electeur) : trivialement dictatorial
    - Induction : clonage d'un electeur, application de l'hypothese de recurrence
    - Full proof: SocialChoice.Impossibilities.GibbardSatterthwaite -/
theorem gibbard_satterthwaite_sketch
    {V A : Type} [DecidableEq A] [Inhabited V] [Inhabited A]
    (f : VotingRule V A)
    (hf_res : IsResolute f)
    (hf_unan : SatisfiesUnanimity f)
    (hf_sp : IsStrategyproof f)
    (h_three : ∃ a b c : A, a ≠ b ∧ b ≠ c ∧ a ≠ c) :
    IsDictatorialRule f := by
  sorry

#check @gibbard_satterthwaite_sketch

-- Dictatorial (for resolute rules): one voter's preference determines the winner
def IsDictatorialRule {V A : Type} (f : VotingRule V A) : Prop :=
                                        ──────────────▶ ❌ Unknown constant `CoeFun`
  ∃ d : V, ∀ P c, f P c → ∀ d' : A, d' ≠ c → (P d).lt c d'
──▶ ❌ expected token

/-- **Gibbard-Satterthwaite (1973/1975)**
    Toute regle resolutive, unanime et non-manipulable (>= 3 candidats)
    est dictatoriale.

    Preuve par induction forte sur le nombre d'electeurs :
    - Base (1 electeur) : trivialement dictatorial
    - Induction : clonage d'un electeur, application de l'hypothese de recurrence
    - Full proof: SocialChoice.Impossibilities.GibbardSatterthwaite -/
theorem gibbard_satterthwaite_sketch
    {V A : Type} [DecidableEq A] [Inhabited V] [Inhabited A]
    (f : VotingRule V A)
    (hf_res : IsResolute f)
    (hf_unan : SatisfiesUnanimity f)
    (hf_sp : IsStrategyproof f)
    (h_three : ∃ a b c : A, a ≠ b ∧ b ≠ c ∧ a ≠ c) :
───────────────▶ ❌ expected token
    IsDictatorialRule f := by
  sorry

#check @gibbard_satterthwaite_sketch
        ────────────────────────────▶ ❌ Unknown identifier `gibbard_satterthwaite_sketch`
--% env 3

Raw input:
{"cmd": "-- Dictatorial (for resolute rules): one voter's preference determines the winner\ndef IsDictatorialRule {V A : Type} (f : VotingRule V A) : Prop :=\n  \u2203 d : V, \u2200 P c, f P c \u2192 \u2200 d' : A, d' \u2260 c \u2192 (P d).lt c d'\n\n/-- **Gibbard-Satterthwaite (1973/1975)**\n    Toute regle resolutive, unanime et non-manipulable (>= 3 candidats)\n    est dictatoriale.\n\n    Preuve par induction forte sur le nombre d'electeurs :\n    - Base (1 electeur) : trivialement dictatorial\n    - Induction : clonage d'un electeur, application de l'hypothese de recurrence\n    - Full proof: SocialChoice.Impossibilities.GibbardSatterthwaite -/\ntheorem gibbard_satterthwaite_sketch\n    {V A : Type} [DecidableEq A] [Inhabited V] [Inhabited A]\n    (f : VotingRule V A)\n    (hf_res : IsResolute f)\n    (hf_unan : SatisfiesUnanimity f)\n    (hf_sp : IsStrategyproof f)\n    (h_three : \u2203 a b c : A, a \u2260 b \u2227 b \u2260 c \u2227 a \u2260 c) :\n    IsDictatorialRule f := by\n  sorry\n\n#check @gibbard_satterthwaite_sketch", "env": 2}
Raw output:
{"messages":
 [{"severity": "error",
   "pos": {"line": 3, "column": 2},
   "endPos": null,
   "data": "expected token"},
  {"severity": "error",
   "pos": {"line": 2, "column": 40},
   "endPos": {"line": 2, "column": 54},
   "data": "Unknown constant `CoeFun`"},
  {"severity": "error",
   "pos": {"line": 19, "column": 15},
   "endPos": null,
   "data": "expected token"},
  {"severity": "error",
   "pos": {"line": 23, "column": 8},
   "endPos": {"line": 23, "column": 36},
   "data": "Unknown identifier `gibbard_satterthwaite_sketch`"}],
 "env": 3}

### Interpretation

Le theoreme de Gibbard-Satterthwaite a des implications profondes :

1. **Toute election est manipulable** : si la regle n'est pas une dictature, il existe toujours une situation ou un electeur a interet a mentir sur ses preferences
2. **La non-manipulabilite est une chimere** : meme les regles les plus sophistiquees (IRV, Borda, Copeland...) sont manipulables
3. **Le choix du systeme de vote est un compromis** : on choisit quelles proprietes sacrifier

### Structure de la preuve (Peters)

La preuve formelle suit l'approche classique par contraposition :
1. On suppose la regle resolutive, unanime, non-manipulable et non-dictatoriale
2. On montre l'existence d'un "pivot" (electeur dont le vote est critique)
3. Par induction sur le nombre d'electeurs, on construit une contradiction

> **Preuve complete** : `SocialChoice.Impossibilities.GibbardSatterthwaite.Main` dans `social_choice_lean_peters/`

---

## 5. Impossibilites de Condorcet

DominikPeters formalise 4 theoremes d'impossibilite lies a Condorcet. Chacun montre que la coherence de Condorcet entre en conflit avec un autre axiome naturel :

1. **Condorcet + Participation** => Impossible (Moulin 1988)
2. **Condorcet + Renforcement** => Impossible (Young 1975)
3. **Condorcet + Non-manipulabilite** => Impossible
4. **Anonymat + Neutralite + Resolutude** => Impossible (nombre pair)

In [5]:
-- Condorcet impossibilities: Condorcet consistency + natural axiom => contradiction
-- All formalized in SocialChoice.Impossibilities.* (Peters project)

-- Key result: if a rule is Condorcet consistent, resolute and strategyproof,
-- it cannot exist (for >= 3 alternatives)
theorem condorcet_strategyproof_impossible_sketch
    {V A : Type} [DecidableEq A]
    (f : VotingRule V A)
    (margin : VotingProfile V A → MarginFun A)
    (hf_res : IsResolute f)
    (hf_cc : CondorcetConsistent f margin)
    (hf_sp : IsStrategyproof f)
    (h_three : ∃ a b c : A, a ≠ b ∧ b ≠ c ∧ a ≠ c) :
    False := by
  sorry

-- No-show paradox: adding voters who rank c first can make c lose
-- Formalized in SocialChoice.Impossibilities.CondorcetParticipation
-- (Moulin 1988)

-- Reinforcement: if disjoint electorates agree, the union should agree
-- Incompatible with Condorcet consistency
-- Formalized in SocialChoice.Impossibilities.CondorcetReinforcement
-- (Young 1975)

#check @condorcet_strategyproof_impossible_sketch

-- Condorcet impossibilities: Condorcet consistency + natural axiom => contradiction
-- All formalized in SocialChoice.Impossibilities.* (Peters project)

-- Key result: if a rule is Condorcet consistent, resolute and strategyproof,
-- it cannot exist (for >= 3 alternatives)
theorem condorcet_strategyproof_impossible_sketch
    {V A : Type} [DecidableEq A]
    (f : VotingRule V A)
    (margin : VotingProfile V A → MarginFun A)
    (hf_res : IsResolute f)
    (hf_cc : CondorcetConsistent f margin)
    (hf_sp : IsStrategyproof f)
    (h_three : ∃ a b c : A, a ≠ b ∧ b ≠ c ∧ a ≠ c) :
───────────────▶ ❌ expected token
    False := by
  sorry

-- No-show paradox: adding voters who rank c first can make c lose
-- Formalized in SocialChoice.Impossibilities.CondorcetParticipation
-- (Moulin 1988)

-- Reinforcement: if disjoint electorates agree, the union should agree
-- Incompatible with Condorcet consistency
-- Formalized in SocialChoice.Impossibilities.CondorcetReinforcement
-- (Young 1975)

#check @condorcet_strategyproof_impossible_sketch
        ─────────────────────────────────────────▶ ❌ Unknown identifier `condorcet_strategyproof_impossible_sketch`
--% env 4

Raw input:
{"cmd": "-- Condorcet impossibilities: Condorcet consistency + natural axiom => contradiction\n-- All formalized in SocialChoice.Impossibilities.* (Peters project)\n\n-- Key result: if a rule is Condorcet consistent, resolute and strategyproof,\n-- it cannot exist (for >= 3 alternatives)\ntheorem condorcet_strategyproof_impossible_sketch\n    {V A : Type} [DecidableEq A]\n    (f : VotingRule V A)\n    (margin : VotingProfile V A \u2192 MarginFun A)\n    (hf_res : IsResolute f)\n    (hf_cc : CondorcetConsistent f margin)\n    (hf_sp : IsStrategyproof f)\n    (h_three : \u2203 a b c : A, a \u2260 b \u2227 b \u2260 c \u2227 a \u2260 c) :\n    False := by\n  sorry\n\n-- No-show paradox: adding voters who rank c first can make c lose\n-- Formalized in SocialChoice.Impossibilities.CondorcetParticipation\n-- (Moulin 1988)\n\n-- Reinforcement: if disjoint electorates agree, the union should agree\n-- Incompatible with Condorcet consistency\n-- Formalized in SocialChoice.Impossibilities.CondorcetReinforcement\n-- (Young 1975)\n\n#check @condorcet_strategyproof_impossible_sketch", "env": 3}
Raw output:
{"messages":
 [{"severity": "error",
   "pos": {"line": 13, "column": 15},
   "endPos": null,
   "data": "expected token"},
  {"severity": "error",
   "pos": {"line": 26, "column": 8},
   "endPos": {"line": 26, "column": 49},
   "data": "Unknown identifier `condorcet_strategyproof_impossible_sketch`"}],
 "env": 4}

### Interpretation des impossibilites de Condorcet

| Impossibilite | Axiomes | Resultat | Reference |
|---------------|---------|----------|-----------|
| Moulin 1988 | Condorcet + Participation | No-show paradox | `CondorcetParticipation` |
| Young 1975 | Condorcet + Renforcement | Incoherence inter-groupes | `CondorcetReinforcement` |
| General | Condorcet + Strategyproof | Manipulable | `CondorcetStrategyproofness` |
| ANR | Anonymat + Neutralite + Resolutude | Impossible (pair) | `AnonymousNeutralResolute` |

Le **no-show paradox** (Moulin 1988) est particulierement frappant : ajouter des electeurs qui classent votre candidat en tete peut le faire perdre ! Cela signifie que ne pas voter peut etre une meilleure strategie que de voter.

Ces resultats montrent que la **coherence de Condorcet** est une condition forte qui exclut de nombreuses proprietes desiderables.

---

## 6. Split Cycle (Holliday & Pacuit 2023)

La regle Split Cycle est un cas particulier dans la theorie du choix social : c'est la regle la plus fine qui soit coherente avec Condorcet et acyclique. Elle resout le probleme des cycles de Condorcet en "affaiblissant" les marges dans les cycles.

In [6]:
-- Split Cycle (Holliday & Pacuit 2023)
-- The finest Condorcet-consistent, acyclic voting rule

-- Simplified: x defeats y if margin(x,y) > 0
-- and no blocking cycle exists (simplified to 3-cycles here)
-- In Peters: arbitrary-length cycles via List A

variable {A : Type}

-- Blocking cycle (simplified to 3-cycles)
-- A 3-cycle z -> x -> y -> z blocks x -> y if margins are >= margin(x,y)
def HasNoBlockingCycle (m : MarginFun A) (x y : A) : Prop :=
  ¬ ∃ z : A, m z x ≥ m x y ∧ m y z ≥ m x y ∧ z ≠ x ∧ z ≠ y

-- Split Cycle defeat: positive margin + no blocking cycle
def SCSimpleDefeat (m : MarginFun A) (x y : A) : Prop :=
  m x y > 0 ∧ HasNoBlockingCycle m x y

-- Split Cycle winners: undefeated alternatives
def SplitCycleWinners (m : MarginFun A) : A → Prop :=
  fun x => ∀ y, ¬ SCSimpleDefeat m y x

#check @HasNoBlockingCycle
#check @SCSimpleDefeat
#check @SplitCycleWinners

-- Split Cycle (Holliday & Pacuit 2023)
-- The finest Condorcet-consistent, acyclic voting rule

-- Simplified: x defeats y if margin(x,y) > 0
-- and no blocking cycle exists (simplified to 3-cycles here)
-- In Peters: arbitrary-length cycles via List A

variable {A : Type}

-- Blocking cycle (simplified to 3-cycles)
-- A 3-cycle z -> x -> y -> z blocks x -> y if margins are >= margin(x,y)
def HasNoBlockingCycle (m : MarginFun A) (x y : A) : Prop :=
                            ───────────▶ ❌ Unknown constant `CoeFun`
  ¬ ∃ z : A, m z x ≥ m x y ∧ m y z ≥ m x y ∧ z ≠ x ∧ z ≠ y
──▶ ❌ expected token

-- Split Cycle defeat: positive margin + no blocking cycle
def SCSimpleDefeat (m : MarginFun A) (x y : A) : Prop :=
                        ───────────▶ ❌ Unknown constant `CoeFun`
  m x y > 0 ∧ HasNoBlockingCycle m x y
────────▶ ❌ expected token

-- Split Cycle winners: undefeated alternatives
def SplitCycleWinners (m : MarginFun A) : A → Prop :=
                           ───────────▶ ❌ Unknown constant `CoeFun`
  fun x => ∀ y, ¬ SCSimpleDefeat m y x
────────────────▶ ❌ expected token

#check @HasNoBlockingCycle
        ──────────────────▶ ❌ Unknown identifier `HasNoBlockingCycle`
#check @SCSimpleDefeat
        ──────────────▶ ❌ Unknown identifier `SCSimpleDefeat`
#check @SplitCycleWinners
        ─────────────────▶ ❌ Unknown identifier `SplitCycleWinners`
--% env 5

Raw input:
{"cmd": "-- Split Cycle (Holliday & Pacuit 2023)\n-- The finest Condorcet-consistent, acyclic voting rule\n\n-- Simplified: x defeats y if margin(x,y) > 0\n-- and no blocking cycle exists (simplified to 3-cycles here)\n-- In Peters: arbitrary-length cycles via List A\n\nvariable {A : Type}\n\n-- Blocking cycle (simplified to 3-cycles)\n-- A 3-cycle z -> x -> y -> z blocks x -> y if margins are >= margin(x,y)\ndef HasNoBlockingCycle (m : MarginFun A) (x y : A) : Prop :=\n  \u00ac \u2203 z : A, m z x \u2265 m x y \u2227 m y z \u2265 m x y \u2227 z \u2260 x \u2227 z \u2260 y\n\n-- Split Cycle defeat: positive margin + no blocking cycle\ndef SCSimpleDefeat (m : MarginFun A) (x y : A) : Prop :=\n  m x y > 0 \u2227 HasNoBlockingCycle m x y\n\n-- Split Cycle winners: undefeated alternatives\ndef SplitCycleWinners (m : MarginFun A) : A \u2192 Prop :=\n  fun x => \u2200 y, \u00ac SCSimpleDefeat m y x\n\n#check @HasNoBlockingCycle\n#check @SCSimpleDefeat\n#check @SplitCycleWinners", "env": 4}
Raw output:
{"messages":
 [{"severity": "error",
   "pos": {"line": 13, "column": 2},
   "endPos": null,
   "data": "expected token"},
  {"severity": "error",
   "pos": {"line": 12, "column": 28},
   "endPos": {"line": 12, "column": 39},
   "data": "Unknown constant `CoeFun`"},
  {"severity": "error",
   "pos": {"line": 16, "column": 24},
   "endPos": {"line": 16, "column": 35},
   "data": "Unknown constant `CoeFun`"},
  {"severity": "error",
   "pos": {"line": 17, "column": 8},
   "endPos": null,
   "data": "expected token"},
  {"severity": "error",
   "pos": {"line": 21, "column": 16},
   "endPos": null,
   "data": "expected token"},
  {"severity": "error",
   "pos": {"line": 20, "column": 27},
   "endPos": {"line": 20, "column": 38},
   "data": "Unknown constant `CoeFun`"},
  {"severity": "error",
   "pos": {"line": 23, "column": 8},
   "endPos": {"line": 23, "column": 26},
   "data": "Unknown identifier `HasNoBlockingCycle`"},
  {"severity": "error",
   "pos": {"line": 24, "column": 8},
   "endPos": {"line": 24, "column": 22},
   "data": "Unknown identifier `SCSimpleDefeat`"},
  {"severity": "error",
   "pos": {"line": 25, "column": 8},
   "endPos": {"line": 25, "column": 25},
   "data": "Unknown identifier `SplitCycleWinners`"}],
 "env": 5}

### Interpretation de Split Cycle

L'idee cle de Split Cycle est d'**affaiblir les relations de defaite** : on ne retient la defaite de x sur y que si aucune marge dans un cycle contenant x et y n'est >= la marge de x sur y.

**Intuition** : si la marge de x sur y est 3, mais qu'il existe un cycle x -> y -> z -> x avec des marges >= 3, alors la defaite de x sur y est "bloquee" par le cycle. On ne peut pas affirmer que x bat y sans affirmer simultanement que y bat z et z bat x.

### Axiomes verifiees pour Split Cycle (Peters)

| Axiome | Statut | Fichier |
|--------|--------|---------|
| Condorcet Consistency | Prouve | `SplitCycle/Condorcet.lean` |
| Monotonicite | Prouve | `SplitCycle/Monotonicity.lean` |
| Pareto | Prouve | `SplitCycle/Pareto.lean` |
| Neutrite | Prouve | `SplitCycle/Neutrality.lean` |
| Smith | Prouve | `SplitCycle/Smith.lean` |
| Clones | Prouve | `SplitCycle/Clones.lean` |
| Independance | Prouve | `SplitCycle/Independence.lean` |
| Renversement | Prouve | `SplitCycle/Reversal.lean` |

Split Cycle est unique : c'est la regle la plus fine satisfaisant Condorcet + acyclicite + une autre propriete naturelle (voir Holliday & Pacuit 2023 pour les details).

---

## 7. Regles de Vote Formalisees

DominikPeters verifie systematiquement les axiomes pour 15+ regles de vote, grace au meta-framework `@[scAxiom]` / `@[scRule]`.

### Tableau comparatif des regles

| Regle | Axiomes verifiees |
|-------|-------------------|
| **Split Cycle** | Condorcet, Monotonicity, Pareto, Neutrality, Smith, Clones, Reversal, Independence |
| **Schulze** | Condorcet, Monotonicity, Neutrality, Smith, Clones, Reversal, Transitivity |
| **Black** | Condorcet, Monotonicity, Neutrality, Pareto, Reversal, Smith |
| **Copeland** | Condorcet, Monotonicity, Neutrality, Pareto, Reversal, Smith |
| **Minimax** | Condorcet, Monotonicity, Neutrality, Pareto, Reversal, Majority |
| **Borda** | Monotonicity, Neutrality, Pareto, Participation, Reinforcement |
| **Plurality** | Monotonicity, Majority |
| **IRV** | CondorcetLoser, Majority, MutualMajority, Clones |
| **Baldwin** | Condorcet, CondorcetLoser, Smith |
| **Coombs** | Condorcet, CondorcetLoser, Majority |
| **Top Cycle** | Condorcet, Monotonicity, Smith, MutualMajority |
| **Uncovered Set** | Condorcet, Monotonicity, Neutrality, Smith, Clones |

**12 regles, 14 axiomes distincts**. Chaque verification est une preuve formelle en Lean 4.

---

## 8. Theoreme de Duggan-Schwartz

Le theoreme de Duggan-Schwartz etend Gibbard-Satterthwaite aux **regles multi-gagnants** (qui peuvent selectionner plusieurs candidats). Il utilise deux notions de non-manipulabilite : optimiste et pessimiste.

In [7]:
-- Duggan-Schwartz: extension to multi-winner rules

-- Optimist strategyproof: can't improve best possible outcome
def IsOptimistSP {V A : Type} [DecidableEq A] (f : VotingRule V A) : Prop :=
  ∀ (P : VotingProfile V A) (v : V) (fake : StrictPref A),
    let P' := fun w : V => if w = v then fake else P w
    ¬ ∃ y, f P' y ∧ ∀ x, f P x → (P v).lt y x

-- Pessimist strategyproof: can't improve worst possible outcome
def IsPessimistSP {V A : Type} [DecidableEq A] (f : VotingRule V A) : Prop :=
  ∀ (P : VotingProfile V A) (v : V) (fake : StrictPref A),
    let P' := fun w : V => if w = v then fake else P w
    ¬ ∃ x, f P x ∧ ∀ y, f P' y → (P v).lt y x

/-- **Duggan-Schwartz (2000)**: A non-trivial, surjective voting rule
    satisfying both optimist and pessimist strategyproofness
    has a "nominating set" (coalition of <= 3 voters controlling outcome).
    Full proof: SocialChoice.Impossibilities.DugganSchwartz -/
theorem duggan_schwartz_sketch
    {V A : Type} [DecidableEq A] [Inhabited V] [Inhabited A]
    (f : VotingRule V A)
    (hf_opt : IsOptimistSP f)
    (hf_pess : IsPessimistSP f)
    (h_three : ∃ a b c : A, a ≠ b ∧ b ≠ c ∧ a ≠ c) :
    ∃ (G : V → Prop), True := by sorry

#check @IsOptimistSP
#check @IsPessimistSP
#check @duggan_schwartz_sketch

-- Duggan-Schwartz: extension to multi-winner rules

-- Optimist strategyproof: can't improve best possible outcome
def IsOptimistSP {V A : Type} [DecidableEq A] (f : VotingRule V A) : Prop :=
  ∀ (P : VotingProfile V A) (v : V) (fake : StrictPref A),
    let P' := fun w : V => if w = v then fake else P w
                          ───▶ ❌ unexpected token 'if'; expected term
    ¬ ∃ y, f P' y ∧ ∀ x, f P x → (P v).lt y x

-- Pessimist strategyproof: can't improve worst possible outcome
def IsPessimistSP {V A : Type} [DecidableEq A] (f : VotingRule V A) : Prop :=
  ∀ (P : VotingProfile V A) (v : V) (fake : StrictPref A),
    let P' := fun w : V => if w = v then fake else P w
                          ───▶ ❌ unexpected token 'if'; expected term
    ¬ ∃ x, f P x ∧ ∀ y, f P' y → (P v).lt y x

/-- **Duggan-Schwartz (2000)**: A non-trivial, surjective voting rule
    satisfying both optimist and pessimist strategyproofness
    has a "nominating set" (coalition of <= 3 voters controlling outcome).
    Full proof: SocialChoice.Impossibilities.DugganSchwartz -/
theorem duggan_schwartz_sketch
    {V A : Type} [DecidableEq A] [Inhabited V] [Inhabited A]
    (f : VotingRule V A)
    (hf_opt : IsOptimistSP f)
    (hf_pess : IsPessimistSP f)
    (h_three : ∃ a b c : A, a ≠ b ∧ b ≠ c ∧ a ≠ c) :
───────────────▶ ❌ expected token
    ∃ (G : V → Prop), True := by sorry

#check @IsOptimistSP
        ────────────▶ ❌ Unknown identifier `IsOptimistSP`
#check @IsPessimistSP
        ─────────────▶ ❌ Unknown identifier `IsPessimistSP`
#check @duggan_schwartz_sketch
        ──────────────────────▶ ❌ Unknown identifier `duggan_schwartz_sketch`
--% env 6

Raw input:
{"cmd": "-- Duggan-Schwartz: extension to multi-winner rules\n\n-- Optimist strategyproof: can't improve best possible outcome\ndef IsOptimistSP {V A : Type} [DecidableEq A] (f : VotingRule V A) : Prop :=\n  \u2200 (P : VotingProfile V A) (v : V) (fake : StrictPref A),\n    let P' := fun w : V => if w = v then fake else P w\n    \u00ac \u2203 y, f P' y \u2227 \u2200 x, f P x \u2192 (P v).lt y x\n\n-- Pessimist strategyproof: can't improve worst possible outcome\ndef IsPessimistSP {V A : Type} [DecidableEq A] (f : VotingRule V A) : Prop :=\n  \u2200 (P : VotingProfile V A) (v : V) (fake : StrictPref A),\n    let P' := fun w : V => if w = v then fake else P w\n    \u00ac \u2203 x, f P x \u2227 \u2200 y, f P' y \u2192 (P v).lt y x\n\n/-- **Duggan-Schwartz (2000)**: A non-trivial, surjective voting rule\n    satisfying both optimist and pessimist strategyproofness\n    has a \"nominating set\" (coalition of <= 3 voters controlling outcome).\n    Full proof: SocialChoice.Impossibilities.DugganSchwartz -/\ntheorem duggan_schwartz_sketch\n    {V A : Type} [DecidableEq A] [Inhabited V] [Inhabited A]\n    (f : VotingRule V A)\n    (hf_opt : IsOptimistSP f)\n    (hf_pess : IsPessimistSP f)\n    (h_three : \u2203 a b c : A, a \u2260 b \u2227 b \u2260 c \u2227 a \u2260 c) :\n    \u2203 (G : V \u2192 Prop), True := by sorry\n\n#check @IsOptimistSP\n#check @IsPessimistSP\n#check @duggan_schwartz_sketch", "env": 5}
Raw output:
{"messages":
 [{"severity": "error",
   "pos": {"line": 6, "column": 26},
   "endPos": {"line": 6, "column": 29},
   "data": "unexpected token 'if'; expected term"},
  {"severity": "error",
   "pos": {"line": 12, "column": 26},
   "endPos": {"line": 12, "column": 29},
   "data": "unexpected token 'if'; expected term"},
  {"severity": "error",
   "pos": {"line": 24, "column": 15},
   "endPos": null,
   "data": "expected token"},
  {"severity": "error",
   "pos": {"line": 27, "column": 8},
   "endPos": {"line": 27, "column": 20},
   "data": "Unknown identifier `IsOptimistSP`"},
  {"severity": "error",
   "pos": {"line": 28, "column": 8},
   "endPos": {"line": 28, "column": 21},
   "data": "Unknown identifier `IsPessimistSP`"},
  {"severity": "error",
   "pos": {"line": 29, "column": 8},
   "endPos": {"line": 29, "column": 30},
   "data": "Unknown identifier `duggan

### Interpretation de Duggan-Schwartz

Le theoreme de Duggan-Schwartz montre que meme les regles multi-gagnants sont vulnerables :

- **Manipulation optimiste** : un electeur peut ameliorer le **meilleur** candidat elu en mentant
- **Manipulation pessimiste** : un electeur peut ameliorer le **pire** candidat elu en mentant
- Si une regle est non-triviale et surjective, elle doit avoir un "nominating set" (petite coalition controlant le resultat)

Ce theoreme est plus general que Gibbard-Satterthwaite : il s'applique meme aux regles qui peuvent eligir plusieurs candidats.

---

**Navigation** : [<< 01-Arrow-Impossibility-Theorem.ipynb](01-Arrow-Impossibility-Theorem.ipynb) | [03-Voting-Methods.ipynb >>](03-Voting-Methods.ipynb) | [Index](../README.md)
